In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

import sys
sys.path.append("..")
sys.path.append("../models")

from models.baseline_Unet_ViT import U_net_ViT


In [ ]:
class MicroCTDataset(Dataset):
    """
    Loads paired TIFF micro-CT images and masks from:
        datas/Original Images/
        datas/Original Masks/

    Assumes:
        image_v2_00.tif
        image_v2_mask_00.tif
    """

    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.img_files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith(".tif")
        ])

        # Generate mask filenames
        self.mask_files = [
            f.replace(".tif", "").replace("image_v2_", "image_v2_mask_") + ".tif"
            for f in self.img_files
        ]

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        # Load TIFF grayscale images
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        img = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        img = T.ToTensor()(img)
        mask = T.ToTensor()(mask)
        mask = (mask > 0.5).float()  # ensure binary

        # Optional augmentations
        if self.transform:
            data = torch.cat([img, mask], dim=0)
            data = self.transform(data)
            img = data[0].unsqueeze(0)
            mask = data[1].unsqueeze(0)

        return img, mask


In [ ]:
img_dir = "../datas/Original Images"
mask_dir = "../datas/Original Masks"

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(10),
])

dataset = MicroCTDataset(img_dir, mask_dir, transform=train_transform)

# 80/20 split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


In [ ]:
class DiceLoss(nn.Module):          #Loss Functions (Dice + BCE)
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        smooth = 1e-6
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        return 1 - (2 * intersection + smooth) / (union + smooth)

criterion = lambda pred, target: (
    nn.BCEWithLogitsLoss()(pred, target) + DiceLoss()(pred, target)
)


In [ ]:
#initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = U_net_ViT(
    encode_in=(1, 64, 128, 256),
    encode_out=(64, 128, 256, 512),
    decode_in=(1024, 512, 256, 128),
    decode_out=(512, 256, 128, 64),
    normalize=True
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
#train
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for img, mask in train_loader:
        img, mask = img.to(device), mask.to(device)

        optimizer.zero_grad()
        pred = model(img)
        _, _, ph, pw = pred.shape                    # crop mask to match pred spatial size 
        mask_cropped = mask[:, :, :ph, :pw]          # crop mask to match pred spatial size 

        loss = criterion(pred, mask_cropped)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.to(device), mask.to(device)
            pred = model(img)
            _, _, ph, pw = pred.shape                 # Crop validation mask to prediction size
            mask = mask[:, :, :ph, :pw]               # Crop validation mask to prediction size
            val_loss += criterion(pred, mask).item()  # Crop validation mask to prediction size

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

# Save model
torch.save(model.state_dict(), "unet_vit_trained.pth")
print("Model saved ✔")


In [ ]:
#Run inference on validation set + save results
import torchvision.transforms.functional as TF

save_dir = "../datas/predictions/"
os.makedirs(save_dir, exist_ok=True)

model.eval()
with torch.no_grad():
    for i, (img, _) in enumerate(val_loader):
        img = img.to(device)
        pred = torch.sigmoid(model(img)).cpu()
        pred_mask = (pred > 0.5).float()

        # Save as image
        out = TF.to_pil_image(pred_mask.squeeze(0))
        out.save(f"{save_dir}/pred_{i:03d}.tif")

print("Predictions saved ✔")


In [ ]:
#disaplay results
sample_img, sample_mask = dataset[0]
model.eval()
with torch.no_grad():
    pred = torch.sigmoid(model(sample_img.unsqueeze(0).to(device))).cpu()
    pred_mask = (pred > 0.5).float().squeeze(0).squeeze(0)

plt.figure(figsize=(10,4))
plt.subplot(1,3,1)
plt.imshow(sample_img.squeeze(), cmap="gray")
plt.title("Input Image")

plt.subplot(1,3,2)
plt.imshow(sample_mask.squeeze(), cmap="gray")
plt.title("Ground Truth Mask")

plt.subplot(1,3,3)
plt.imshow(pred_mask, cmap="gray")
plt.title("Predicted Mask")

plt.show()


In [2]:
import torch
print(torch.cuda.is_available())



False
